## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-23-14-54-08 +0000,nyt,Don Toliver Concert in Manchester Is Evacuated...,https://www.nytimes.com/2026/11/23/world/europ...
1,2026-11-23-14-54-03 +0000,bbc,Mexican soldiers attacked in wake of drug lord...,https://www.bbc.co.uk/news/articles/c5rmgkv2lk...
2,2026-11-23-14-51-55 +0000,nypost,Poland says Russia allegedly twice tried to ki...,https://nypost.com/2026/11/23/world-news/polan...
3,2026-11-23-14-51-05 +0000,bbc,US government reassured Nvidia board over Huang,https://www.bbc.co.uk/news/videos/cqze7lwdglng...
4,2026-11-23-14-51-00 +0000,wsj,Nvidia Board Reinstates Jensen Huang as CEO,https://www.wsj.com/articles/nvidia-board-rein...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_3124/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
25,nvidia,18
93,deal,16
27,huang,16
65,trump,15
378,new,12


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
127,2026-11-23-09-52-53 +0000,wsj,Stock Market Today: Nvidia Stock Surges as Inv...,https://www.wsj.com/livecoverage/stock-market-...,85
12,2026-11-23-14-37-22 +0000,nypost,Trump hails Nvidia’s ‘correct and proper’ move...,https://nypost.com/2026/11/23/us-news/trump-ha...,80
129,2026-11-23-09-30-17 +0000,nypost,Jensen Huang’s lover was in contact with Chine...,https://nypost.com/2026/11/23/us-news/jensen-h...,79
32,2026-11-23-14-06-27 +0000,nypost,Nvidia reinstates Jensen Huang as CEO weeks af...,https://nypost.com/2026/11/23/business/nvidia-...,74
154,2026-11-23-05-44-29 +0000,nypost,Trump says he hopes Nvidia rehires Jensen Huan...,https://nypost.com/2026/11/23/us-news/trump-sa...,71


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
127,85,2026-11-23-09-52-53 +0000,wsj,Stock Market Today: Nvidia Stock Surges as Inv...,https://www.wsj.com/livecoverage/stock-market-...
132,61,2026-11-23-09-17-10 +0000,nypost,Zelensky confirms Ukraine was behind drone att...,https://nypost.com/2026/11/23/world-news/zelen...
269,53,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...
122,53,2026-11-23-10-00-00 +0000,wsj,Democrats making presidential bids will gather...,https://www.wsj.com/politics/elections/democra...
256,51,2026-11-22-17-11-00 +0000,cbc,COP31 climate talks collapse after member stat...,https://www.cbc.ca/news/world/cop31-talks-coll...
185,47,2026-11-23-00-00-00 +0000,wsj,Incoming Democratic Sen.-elect Abdul El-Sayed ...,https://www.wsj.com/politics/abdul-el-sayed-is...
18,44,2026-11-23-14-28-00 +0000,wsj,The U.S. Army said it plans to detonate severa...,https://www.wsj.com/politics/national-security...
229,43,2026-11-22-19-41-02 +0000,wapo,Democrat Chuck Hubbard defeats GOP incumbent i...,https://www.washingtonpost.com/politics/2026/1...
213,34,2026-11-22-21-03-28 +0000,latimes,Mother sues California school district after s...,https://www.latimes.com/california/story/2026-...
72,34,2026-11-23-12-12-18 +0000,nypost,Former Pete Hegseth chief of staff calls on wa...,https://nypost.com/2026/11/23/us-news/former-p...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
